# Lab 5 - Safe reload
Checks that reruns and replays do not duplicate or lose data. The `action` widget selects a step:
- `capture` saves the Kafka keys (topic, partition, offset) of Wikipedia Bronze.
- `check_unchanged` compares them after a rerun without new events: Bronze must be identical.
- `check_growth` expects more rows after new events, with no key repeated and no old key lost.
- `check_replay` expects Silver and quarantine, rebuilt by a full refresh, to equal a recomputation from the saved Bronze.

Bronze has `pipelines.reset.allowed=false`, since Event Hub drops old events. Only the downstream tables are refreshed.

In [ ]:
%run ./lab5_00_config

In [ ]:
dbutils.widgets.dropdown("action", "capture", ["capture", "check_unchanged", "check_growth", "check_replay"])
action = dbutils.widgets.get("action")
baseline_table = table("reload_baseline")
keys = ["kafka_topic", "kafka_partition", "kafka_offset"]
if action == "capture":
    version = delta_version("wiki_bronze")
    bronze = read_version("wiki_bronze", version)
    assert bronze.limit(1).count() > 0
    bronze.select(*keys).write.format("delta").mode("overwrite").saveAsTable(baseline_table)
    print("Captured delivery keys; now stop input and run the next pipeline update.")
else:
    baseline = spark.table(baseline_table)
    current = spark.table(table("wiki_bronze"))
    assert baseline.exceptAll(current.select(*keys)).limit(1).count() == 0, "Raw history was lost"
    assert current.groupBy(*keys).count().filter("count > 1").limit(1).count() == 0
    if action in ["check_unchanged", "check_replay"]:
        assert_equal(current, baseline, keys, "raw Bronze unchanged")
    if action == "check_growth":
        assert current.count() > baseline.count(), "No newly ingested events"
    if action == "check_replay":
        expected = annotate_quality(parse_wiki(current), WIKI_RULES)
        assert_equal(spark.table(table("wiki_silver")), expected.filter("_is_valid"), WIKI_BUSINESS_COLUMNS + keys, "Silver recomputed from preserved Bronze")
        assert_equal(spark.table(table("wiki_quarantine")), expected.filter("NOT _is_valid"), keys + ["_quality_errors"], "quarantine recomputed")
    print("PASS:", action)